# LangChain Financial Agent Notebook

This notebook is the study/demo version of `main.py`.

It demonstrates an end-to-end workflow using:
- LangChain orchestration
- Tool calling with yfinance
- Structured outputs
- Context/token optimization
- Memory persistence
- Grounding diagnostics

In [1]:
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import yfinance as yf
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Configuration
Set your ticker, question, and Ollama model name.

In [2]:
TICKER = 'NVDA'
QUESTION = 'What is the latest financial picture for this stock?'
MODEL_NAME = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434'
TOKEN_BUDGET = 700
MAX_SNIPPETS = 8

BASE_DIR = Path.cwd()
MEMORY_DIR = BASE_DIR / 'memory'
MEMORY_DIR.mkdir(parents=True, exist_ok=True)

## Utilities and schema

In [3]:
class StructuredFinancialAnswer(BaseModel):
    ticker: str = Field(description='Ticker symbol')
    question: str = Field(description='User question')
    short_answer: str = Field(description='Concise answer')
    key_points: list[str] = Field(description='Main grounded points')
    risks: list[str] = Field(description='Uncertainties or risk notes')
    evidence_used: list[str] = Field(description='Evidence snippets used')
    confidence: float = Field(description='0.0 to 1.0 confidence')

def normalize_tokens(text: str) -> set[str]:
    return set(re.findall(r'[a-z0-9]+', text.lower()))

def estimate_tokens(text: str) -> int:
    return max(1, int(len(text.split()) * 1.1))

def memory_path(ticker: str) -> Path:
    return MEMORY_DIR / f'{ticker.upper()}.json'

def load_memory(ticker: str, limit: int = 3) -> list[dict[str, Any]]:
    path = memory_path(ticker)
    if not path.exists():
        return []
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
        return data[:limit] if isinstance(data, list) else []
    except Exception:
        return []

def save_memory(ticker: str, entry: dict[str, Any], max_entries: int = 20) -> None:
    entries = load_memory(ticker, limit=max_entries)
    entries.insert(0, entry)
    memory_path(ticker).write_text(
        json.dumps(entries[:max_entries], indent=2, ensure_ascii=True),
        encoding='utf-8',
    )

## Tool calling: yfinance snapshot tool

In [4]:
def extract_latest_close(hist) -> float | None:
    if hist is None or hist.empty:
        return None
    close_col = hist.get('Close')
    if close_col is None:
        return None
    clean = close_col.dropna()
    if clean.empty:
        return None
    return float(clean.iloc[-1])

@tool('get_stock_snapshot')
def get_stock_snapshot(ticker: str) -> dict[str, Any]:
    """Fetch latest stock snapshot, recent OHLCV, and headlines from yfinance."""
    t = yf.Ticker(ticker.upper())
    hist = t.history(period='5d', interval='1d')
    latest_close = extract_latest_close(hist)

    rows = []
    if hist is not None and not hist.empty:
        for idx, row in hist.tail(5).iterrows():
            rows.append({
                'date': str(idx.date()),
                'open': float(row.get('Open', 0.0)),
                'high': float(row.get('High', 0.0)),
                'low': float(row.get('Low', 0.0)),
                'close': float(row.get('Close', 0.0)),
                'volume': int(row.get('Volume', 0)),
            })

    info = {}
    try:
        fi = getattr(t, 'fast_info', None)
        if fi:
            info = {
                'currency': fi.get('currency'),
                'exchange': fi.get('exchange'),
                'market_cap': fi.get('market_cap'),
                'day_high': fi.get('day_high'),
                'day_low': fi.get('day_low'),
            }
    except Exception as exc:
        info = {'error': f'fast_info unavailable: {exc}'}

    headlines = []
    try:
        for item in (getattr(t, 'news', []) or [])[:5]:
            title = item.get('title')
            if title:
                headlines.append(str(title))
    except Exception as exc:
        headlines.append(f'news unavailable: {exc}')

    return {
        'ticker': ticker.upper(),
        'latest_close': latest_close,
        'snapshot_time_utc': datetime.now(timezone.utc).isoformat(),
        'info': info,
        'recent_ohlcv': rows,
        'headlines': headlines,
    }

## Token optimization helpers
Retrieve broadly, rerank, then pass minimal context.

In [5]:
def build_evidence_snippets(snapshot: dict[str, Any]) -> list[str]:
    snippets = []
    snippets.append(
        f"Ticker {snapshot.get('ticker')} latest_close={snapshot.get('latest_close')} snapshot_time_utc={snapshot.get('snapshot_time_utc')}"
    )

    info = snapshot.get('info', {}) or {}
    snippets.append(
        f"Info currency={info.get('currency')} exchange={info.get('exchange')} market_cap={info.get('market_cap')} day_high={info.get('day_high')} day_low={info.get('day_low')}"
    )

    for row in snapshot.get('recent_ohlcv', [])[:5]:
        snippets.append(
            f"OHLCV date={row.get('date')} open={row.get('open')} high={row.get('high')} low={row.get('low')} close={row.get('close')} volume={row.get('volume')}"
        )

    for title in snapshot.get('headlines', [])[:5]:
        snippets.append(f'Headline: {title}')

    return snippets

def rerank_snippets(question: str, snippets: list[str]) -> list[tuple[float, str]]:
    q_terms = normalize_tokens(question)
    ranked = []
    for snippet in snippets:
        s_terms = normalize_tokens(snippet)
        overlap = len(q_terms & s_terms)
        coverage = overlap / max(1, len(q_terms))
        ranked.append((overlap + coverage, snippet))
    ranked.sort(key=lambda x: x[0], reverse=True)
    return ranked

def select_minimal_context(ranked: list[tuple[float, str]], token_budget: int, max_snippets: int) -> tuple[list[str], int]:
    selected = []
    used = 0
    for _, snippet in ranked:
        t = estimate_tokens(snippet)
        if selected and used + t > token_budget:
            continue
        selected.append(snippet)
        used += t
        if len(selected) >= max_snippets or used >= token_budget:
            break
    return selected, used

## Orchestration run

In [6]:
llm = ChatOllama(model=MODEL_NAME, base_url=OLLAMA_BASE_URL, temperature=0)
planner = llm.bind_tools([get_stock_snapshot])

tool_message = planner.invoke([
    SystemMessage(content='For stock questions, call get_stock_snapshot first. Prefer tool calls over guessing.'),
    HumanMessage(content=f'Ticker: {TICKER}\nQuestion: {QUESTION}'),
])

tool_calls = tool_message.tool_calls or []
snapshots = []

if tool_calls:
    for call in tool_calls:
        if call.get('name') == 'get_stock_snapshot':
            args = call.get('args', {}) or {}
            snapshots.append(get_stock_snapshot.invoke({'ticker': str(args.get('ticker', TICKER))}))

if not snapshots:
    snapshots.append(get_stock_snapshot.invoke({'ticker': TICKER}))

broad_snippets = []
for s in snapshots:
    broad_snippets.extend(build_evidence_snippets(s))

ranked = rerank_snippets(QUESTION, broad_snippets)
selected_snippets, used_tokens = select_minimal_context(ranked, TOKEN_BUDGET, MAX_SNIPPETS)

print('Tool calls:', len(tool_calls))
print('Broad snippets:', len(broad_snippets))
print('Selected snippets:', len(selected_snippets))
print('Estimated context tokens:', used_tokens)

Tool calls: 1
Broad snippets: 7
Selected snippets: 7
Estimated context tokens: 45


## Structured synthesis and grounding checks

In [ ]:
def grounding_check(answer_text: str, evidence: list[str]) -> dict[str, Any]:
    ans_terms = normalize_tokens(answer_text)
    evidence_terms = set()
    for e in evidence:
        evidence_terms |= normalize_tokens(e)
    unsupported = sorted(term for term in ans_terms if term not in evidence_terms)
    ratio = 1.0 - (len(unsupported) / max(1, len(ans_terms)))
    return {
        'supported_ratio': round(ratio, 3),
        'unsupported_terms_sample': unsupported[:12],
        'status': 'pass' if ratio >= 0.72 else 'review',
    }

parser = PydanticOutputParser(pydantic_object=StructuredFinancialAnswer)
format_instructions = parser.get_format_instructions()
prior_memory = load_memory(TICKER)

prompt = (
    'Use only provided evidence and memory. If uncertain, state uncertainty clearly.\n'
    f'{format_instructions}\n\n'
    f'Ticker: {TICKER}\nQuestion: {QUESTION}\n\n'
    f'Memory:\n{json.dumps(prior_memory, ensure_ascii=True, indent=2)}\n\n'
    f'Evidence context:\n' + '\n'.join(f'- {s}' for s in selected_snippets)
)

raw = llm.invoke([
    SystemMessage(content='Return only structured JSON matching the schema.'),
    HumanMessage(content=prompt),
])

parsed = parser.parse(raw.content)
grounding = grounding_check(parsed.short_answer + ' ' + ' '.join(parsed.key_points), selected_snippets)

save_memory(TICKER, {
    'ts_utc': datetime.now(timezone.utc).isoformat(),
    'question': QUESTION,
    'short_answer': parsed.short_answer,
    'key_points': parsed.key_points,
    'grounding': grounding,
})

output = {
    'result': parsed.model_dump(),
    'diagnostics': {
        'ticker': TICKER,
        'model': MODEL_NAME,
        'selected_snippet_count': len(selected_snippets),
        'estimated_context_tokens': used_tokens,
        'token_budget': TOKEN_BUDGET,
        'grounding': grounding,
    },
}

print(json.dumps(output, indent=2, ensure_ascii=True))